In [17]:
import vitaldb
import pandas as pd
import numpy as np

from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

In [18]:
TRACKS = [
    "SNUADC/ECG_II",
    "SNUADC/PLETH",
    "Solar8000/HR",
    "Solar8000/PLETH_SPO2",
    "Solar8000/ART_MBP"
]

caseids = sorted(
    list(
        vitaldb.find_cases(TRACKS)
    )
)


print("Cases:", len(caseids))

/tmp/ipykernel_609541/3364289593.py:11: DeprecationWarning: find_cases() relies on the per-track 'trks' index, which is deprecated and may be removed in a future release.
  vitaldb.find_cases(TRACKS)


Cases: 3506


In [19]:
def generate_windows(
    df,
    input_minutes=15,
    prediction_minutes=30,
    stride_minutes=5
):

    input_seconds = input_minutes * 60
    prediction_seconds = prediction_minutes * 60
    stride_seconds = stride_minutes * 60

    windows = []

    total_needed = (
        input_seconds +
        prediction_seconds
    )

    for start in range(
        0,
        len(df) - total_needed + 1,
        stride_seconds
    ):

        input_df = df.iloc[
            start:
            start + input_seconds
        ]

        prediction_df = df.iloc[
            start + input_seconds:
            start + input_seconds + prediction_seconds
        ]

        windows.append(
            (
                input_df,
                prediction_df
            )
        )

    return windows

In [22]:
def create_labels(prediction_df):

    hr = prediction_df[
        "Solar8000/HR"
    ].dropna()

    spo2 = prediction_df[
        "Solar8000/PLETH_SPO2"
    ].dropna()

    map_ = prediction_df[
        "Solar8000/ART_MBP"
    ].dropna()

    tachycardia = int(
        len(hr) > 0 and
        hr.max() > 110
    )

    hypotension = int(
        len(map_) > 0 and
        map_.min() < 65
    )

    hypoxia = int(
        len(spo2) > 0 and
        spo2.min() < 90
    )

    return {
        "tachycardia": tachycardia,
        "hypotension": hypotension,
        "hypoxia": hypoxia
    }

In [23]:
def process_case(caseid):

    try:

        arr = vitaldb.load_case(
            caseid,
            TRACKS,
            interval=1
        )

        df = pd.DataFrame(
            arr,
            columns=TRACKS
        )

        # clean invalid values

        df["Solar8000/HR"] = (
            df["Solar8000/HR"]
            .mask(
                df["Solar8000/HR"] <= 0
            )
        )

        df["Solar8000/PLETH_SPO2"] = (
            df["Solar8000/PLETH_SPO2"]
            .mask(
                df["Solar8000/PLETH_SPO2"] <= 0
            )
        )

        df["Solar8000/ART_MBP"] = (
            df["Solar8000/ART_MBP"]
            .mask(
                df["Solar8000/ART_MBP"] <= 0
            )
        )

        windows = generate_windows(df)

        rows = []

        for idx, (
            input_df,
            prediction_df
        ) in enumerate(windows):

            labels = create_labels(
                prediction_df
            )

            rows.append({
                "caseid": caseid,
                "window_id": idx,
                "tachycardia":
                    labels["tachycardia"],
                "hypotension":
                    labels["hypotension"],
                "hypoxia":
                    labels["hypoxia"]
            })

        return rows

    except Exception:

        return []

In [24]:
dataset = pd.read_csv("window_dataset.csv")

for col in [
    "tachycardia",
    "hypotension",
    "hypoxia"
]:
    print("\n", col)
    print(dataset[col].value_counts())


 tachycardia
tachycardia
0    44053
1    18205
Name: count, dtype: int64

 hypotension
hypotension
1    34189
0    28069
Name: count, dtype: int64

 hypoxia
hypoxia
0    56632
1     5626
Name: count, dtype: int64


In [25]:
print("caseids:", len(caseids))

df = pd.read_csv("window_dataset.csv")
print("dataset cases:", df["caseid"].nunique())

processed = pd.read_csv("processed_caseids_new.csv")
print("processed cases:", processed["caseid"].nunique())

caseids: 3506
dataset cases: 1761
processed cases: 1761


In [26]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
import os
import time

N_WORKERS = 8

OUTPUT_FILE = "window_dataset.csv"
PROCESSED_FILE = "processed_caseids_new.csv"

# ==================================================
# RESUME SUPPORT
# ==================================================

if os.path.exists(PROCESSED_FILE):

    processed_ids = set(
        pd.read_csv(
            PROCESSED_FILE
        )["caseid"].tolist()
    )

    print(
        f"Found {len(processed_ids)} processed cases"
    )

else:

    processed_ids = set()

remaining_caseids = [
    c for c in caseids
    if c not in processed_ids
]

print(
    f"Remaining cases: {len(remaining_caseids)}"
)

# ==================================================
# STATS
# ==================================================

success_cases = 0
failed_cases = 0
total_windows = 0

start_time = time.time()

print("=" * 60)
print("Starting VitalDB Dataset Generation")
print(f"Total Cases      : {len(caseids)}")
print(f"Already Finished : {len(processed_ids)}")
print(f"Remaining Cases  : {len(remaining_caseids)}")
print(f"Workers          : {N_WORKERS}")
print("=" * 60)

first_write = not os.path.exists(
    OUTPUT_FILE
)

# ==================================================
# PARALLEL EXECUTION
# ==================================================

with ThreadPoolExecutor(
    max_workers=N_WORKERS
) as executor:

    futures = {
        executor.submit(
            process_case,
            caseid
        ): caseid
        for caseid in remaining_caseids
    }

    pbar = tqdm(
        total=len(remaining_caseids),
        desc="Cases",
        unit="case"
    )

    for future in as_completed(futures):

        caseid = futures[future]

        try:

            rows = future.result()

            if len(rows) > 0:

                chunk = pd.DataFrame(rows)

                # ==========================
                # SAVE DATASET IMMEDIATELY
                # ==========================

                chunk.to_csv(
                    OUTPUT_FILE,
                    mode="w" if first_write else "a",
                    header=first_write,
                    index=False
                )

                first_write = False

                # ==========================
                # SAVE CASE ID IMMEDIATELY
                # ==========================

                pd.DataFrame({
                    "caseid": [caseid]
                }).to_csv(
                    PROCESSED_FILE,
                    mode="a",
                    header=not os.path.exists(
                        PROCESSED_FILE
                    ),
                    index=False
                )

                success_cases += 1
                total_windows += len(rows)

                print(
                    f"\n✓ Case {caseid:<5} | "
                    f"Windows: {len(rows):<4} | "
                    f"Saved"
                )

            else:

                failed_cases += 1

                print(
                    f"\n⚠ Case {caseid} "
                    f"returned no windows"
                )

        except Exception as e:

            failed_cases += 1

            print(
                f"\n❌ Case {caseid} FAILED"
            )

            print(
                f"Reason: {str(e)}"
            )

        pbar.set_postfix({
            "Success": success_cases,
            "Failed": failed_cases,
            "Windows": total_windows
        })

        pbar.update(1)

    pbar.close()

end_time = time.time()

# ==================================================
# SUMMARY
# ==================================================

print("\n" + "=" * 60)
print("PROCESSING COMPLETE")
print("=" * 60)

print(
    f"Successful Cases : {success_cases}"
)

print(
    f"Failed Cases     : {failed_cases}"
)

print(
    f"Total Windows    : {total_windows}"
)

print(
    f"Time Taken       : "
    f"{(end_time-start_time)/60:.2f} min"
)

print("=" * 60)

Found 1761 processed cases
Remaining cases: 1745
Starting VitalDB Dataset Generation
Total Cases      : 3506
Already Finished : 1761
Remaining Cases  : 1745
Workers          : 8


Cases:   0%|          | 1/1745 [00:03<1:35:37,  3.29s/case, Success=0, Failed=1, Windows=0]


⚠ Case 1147 returned no windows


Cases:   0%|          | 2/1745 [00:05<1:19:18,  2.73s/case, Success=0, Failed=2, Windows=0]


⚠ Case 1524 returned no windows


...


✓ Case 6351  | Windows: 94   | Saved


Cases: 100%|█████████▉| 1744/1745 [55:20<00:01,  1.81s/case, Success=1732, Failed=12, Windows=62500]


✓ Case 6378  | Windows: 75   | Saved


Cases: 100%|██████████| 1745/1745 [55:23<00:00,  1.90s/case, Success=1733, Failed=12, Windows=62569]


✓ Case 6383  | Windows: 69   | Saved

PROCESSING COMPLETE
Successful Cases : 1733
Failed Cases     : 12
Total Windows    : 62569
Time Taken       : 55.40 min


In [28]:
dataset = pd.read_csv(
    "window_dataset.csv"
)

processed_ids = pd.read_csv(
    "processed_caseids_new.csv"
)

print("Dataset Shape:")
print(dataset.shape)

print(
    "\nProcessed Cases:"
)

print(
    len(processed_ids)
)

dataset.head()

Dataset Shape:
(124827, 5)

Processed Cases:
3494


,caseid,window_id,tachycardia,hypotension,hypoxia
0,3,0,0,0,0
1,3,1,0,0,0
2,3,2,0,0,0
3,3,3,1,0,0
4,3,4,1,0,0


In [29]:
dataset.to_parquet(
    "window_dataset.parquet",
    index=False
)

print("Saved:")
print("window_dataset.csv")
print("window_dataset.parquet")
print("processed_caseids.csv")

Saved:
window_dataset.csv
window_dataset.parquet
processed_caseids.csv


In [30]:
import pandas as pd

df = pd.read_csv("window_dataset.csv")

print(df.shape)
print(df.head())

print("Cases:", df.caseid.nunique())

print("Tachycardia rate :", df.tachycardia.mean())
print("Hypotension rate :", df.hypotension.mean())
print("Hypoxia rate     :", df.hypoxia.mean())

(124827, 5)
   caseid  window_id  tachycardia  hypotension  hypoxia
0       3          0            0            0        0
1       3          1            0            0        0
2       3          2            0            0        0
3       3          3            1            0        0
4       3          4            1            0        0
Cases: 3494
Tachycardia rate : 0.28965688512901855
Hypotension rate : 0.544257252036819
Hypoxia rate     : 0.08795372795949594
